In [ ]:
!pip install mediapipe gradio
import mediapipe as mp
import cv2
import numpy as np
import gradio as gr

# Drawing helpers
mp_drawing = mp.solutions.drawing_utils
mp_pose = mp.solutions.pose
pose = mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5)

In [ ]:
def calculate_angle(a, b, c):
    a, b, c = np.array(a), np.array(b), np.array(c)

    radians = np.arctan2(c[1] - b[1], c[0] - b[0]) - np.arctan2(a[1] - b[1], a[0] - b[0])
    angle = np.abs(radians * 180.0 / np.pi)

    if angle > 180.0:
        angle = 360 - angle

    return angle

In [ ]:
def check_pullup_feedback(landmarks):
    # Left Arm Angle (Shoulder-Elbow-Wrist)
    left_shoulder = [landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].x, landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].y]
    left_elbow = [landmarks[mp_pose.PoseLandmark.LEFT_ELBOW.value].x, landmarks[mp_pose.PoseLandmark.LEFT_ELBOW.value].y]
    left_wrist = [landmarks[mp_pose.PoseLandmark.LEFT_WRIST.value].x, landmarks[mp_pose.PoseLandmark.LEFT_WRIST.value].y]

    # Right Arm Angle
    right_shoulder = [landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].x, landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].y]
    right_elbow = [landmarks[mp_pose.PoseLandmark.RIGHT_ELBOW.value].x, landmarks[mp_pose.PoseLandmark.RIGHT_ELBOW.value].y]
    right_wrist = [landmarks[mp_pose.PoseLandmark.RIGHT_WRIST.value].x, landmarks[mp_pose.PoseLandmark.RIGHT_WRIST.value].y]

    left_angle = calculate_angle(left_shoulder, left_elbow, left_wrist)
    right_angle = calculate_angle(right_shoulder, right_elbow, right_wrist)
    avg_elbow_angle = (left_angle + right_angle) / 2

    # --- Feedback Logic ---

    # 1. Full Extension Check (Angle near 170-180 degrees)
    if avg_elbow_angle > 165:
        feedback = "Hanging: Ensure Full Extension at Bottom"
    # 2. Top Position Check (Angle near 40-50 degrees)
    elif avg_elbow_angle < 60:
        feedback = "Perfect Pull-Up! Chin is Up"
    # 3. Mid-Rep/Insufficient Range
    else:
        feedback = "Mid-Rep: Go Higher or Lower!"

    # --- Accuracy Calculation ---
    # Check distance to Top (45 deg)
    dist_to_top = abs(avg_elbow_angle - 45)
    # Check distance to Bottom (175 deg)
    dist_to_bottom = abs(avg_elbow_angle - 175)

    # Use the distance to the closest target range for scoring
    if dist_to_top < dist_to_bottom:
        # Closer to the top, score based on top form
        accuracy = max(0, 100 - dist_to_top * 1.5)
    else:
        # Closer to the bottom, score based on bottom form
        accuracy = max(0, 100 - dist_to_bottom * 1.5)

    accuracy = max(0, min(100, int(accuracy)))

    return feedback, int(accuracy)

In [ ]:
def analyze_pullups(video_path):
    # Remember to initialize pose object in Block 1 if restarting kernel:
    # pose = mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5)

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return "Error: Could not open input video."

    frame_width, frame_height = int(cap.get(3)), int(cap.get(4))

    # Fix for video speed (stable FPS)
    fps = 30

    # Robust Codec Fix: Use 'MJPG' and .avi extension
    fourcc = cv2.VideoWriter_fourcc(*'MJPG')
    output_video = "output_pullups.avi"
    out = cv2.VideoWriter(output_video, fourcc, fps, (frame_width, frame_height))

    if not out.isOpened():
        return "Error: Could not create output video writer."

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # Fix for mirrored video (un-mirroring webcam view)
        frame = cv2.flip(frame, 1)

        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = pose.process(image)
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

        if results.pose_landmarks:
            landmarks = results.pose_landmarks.landmark

            # Check visibility of a critical landmark before calculation
            if landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].visibility > 0.6:
                mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_pose.POSE_CONNECTIONS)

                feedback, accuracy = check_pullup_feedback(landmarks)

                color = (0, 255, 0) if "Perfect" in feedback else (0, 0, 255)

                cv2.putText(image, feedback, (50, 50), cv2.FONT_HERSHEY_COMPLEX, 1, color, 3)
                cv2.putText(image, f"Accuracy: {accuracy}%", (50, 100), cv2.FONT_HERSHEY_COMPLEX, 1, color, 3)

            else:
                 cv2.putText(image, "NO POSE DETECTED - Adjust Position", (50, 50), cv2.FONT_HERSHEY_COMPLEX, 1, (255, 255, 255), 2)

        else:
             cv2.putText(image, "NO POSE DETECTED - Adjust Position", (50, 50), cv2.FONT_HERSHEY_COMPLEX, 1, (255, 255, 255), 2)


        out.write(image)

    cap.release()
    out.release()
    return output_video

# Gradio Interface
gr.Interface(
    fn=analyze_pullups,
    inputs=gr.Video(sources=["upload", "webcam"]),
    outputs=gr.Video(),
    title="Pull-Up Form Analyzer",
    description="Upload a video of your pull-ups, and get feedback on your range of motion!",
).launch(share=True, debug=True)